## Escenario 2: Equipo colaborativo con MLflow Tracking Server local

MLflow setup:
- Tracking server: **sí** (local)
- Backend store: **SQLite** (archivo local)
- Artifacts store: **filesystem local**
- Model Registry: **disponible** (útil para aprender el flujo), pero para producción se recomienda backend DB + artifact store remoto (ver Scenario 3)

Antes de ejecutar este notebook, inicia el servidor MLflow (desde la raíz del proyecto):

```bash
uv run mlflow server \
  --host 127.0.0.1 \
  --port 5000 \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns \
  --allowed-hosts "localhost,127.0.0.1,127.0.0.1:5000"
```

> **Nota (MLflow >= 3.5):** desde esta versión, el servidor valida el header `Host` de cada request por defecto (protección contra DNS rebinding). Normalmente `localhost`/`127.0.0.1` ya están permitidos, pero si tu máquina pasa por una VPN, Docker, WSL o algún proxy que reescribe ese header, puedes ver un error `403` como `API request to endpoint /api/2.0/mlflow/experiments/search failed with error code 403`. El flag `--allowed-hosts` de arriba lo deja explícito y evita el problema.

Luego abre la UI en: http://127.0.0.1:5000

## Configuracion del tracking URI

Conecta el cliente MLflow al servidor local en el puerto 5000.


In [10]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://127.0.0.1:5000'


Confirma que la conexión al servidor se estableció correctamente.

## Listado de Experimentos Disponibles

In [11]:
mlflow.search_experiments()

[<Experiment: artifact_location='/Users/mdurango/Downloads/proyectos/MLOps_UdM/02-Experiment-Tracking/scenarios/mlruns/0', creation_time=1789327674894, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1789327674894, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

## Ejecución del Experimento

In [12]:
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_experiment("iris-server-local")

with mlflow.start_run(run_name="logreg_baseline"):
    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42, "max_iter": 200}
    mlflow.log_params(params)

    mlflow.set_tags(
        {
            "scenario": "2_local_tracking_server",
            "developer": "Maria Durango",
            "module": "mlops_tracking",
            "model_family": "logistic_regression",
            "dataset": "iris",
        }
    )

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)

    mlflow.log_metric("accuracy", float(accuracy_score(y, y_pred)))
    mlflow.log_metric("f1_macro", float(f1_score(y, y_pred, average="macro")))

    preds_path = "predictions_logreg.csv"
    pd.DataFrame({"y_true": y, "y_pred": y_pred}).to_csv(preds_path, index=False)
    mlflow.log_artifact(preds_path)

    model_info = mlflow.sklearn.log_model(
        lr,
        name="logreg_baseline",
        input_example=X[[0]],
        registered_model_name="iris-classifier",
    )

    print({"model_uri": model_info.model_uri, "artifact_uri": mlflow.get_artifact_uri()})

2026/09/13 14:28:00 INFO mlflow.tracking.fluent: Experiment with name 'iris-server-local' does not exist. Creating a new experiment.
2026/09/13 14:28:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'iris-classifier'.
2026/09/13 14:28:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1


{'model_uri': 'models:/m-cae2614ce96847d4b2cbdef50f0cde31', 'artifact_uri': '/Users/mdurango/Downloads/proyectos/MLOps_UdM/02-Experiment-Tracking/scenarios/mlruns/1/4b1cd6e8e47143b8a6d01f8821026937/artifacts'}
🏃 View run logreg_baseline at: http://127.0.0.1:5000/#/experiments/1/runs/4b1cd6e8e47143b8a6d01f8821026937
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '1' of model 'iris-classifier'.


## Resultados de la Ejecución

La ejecución genera:
- Un URI único para los artifacts
- Enlaces directos al servidor MLflow para visualizar:
    - La ejecución específica
    - El experimento completo

Ahora debería aparecer:
- "Default" (creado automáticamente)
- "iris-server-local" (nuestro experimento)

In [13]:
mlflow.search_experiments()

[<Experiment: artifact_location='/Users/mdurango/Downloads/proyectos/MLOps_UdM/02-Experiment-Tracking/scenarios/mlruns/1', creation_time=1789327680684, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789327680684, lifecycle_stage='active', name='iris-server-local', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/Users/mdurango/Downloads/proyectos/MLOps_UdM/02-Experiment-Tracking/scenarios/mlruns/0', creation_time=1789327674894, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1789327674894, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

### Interactuando con el model registry

Crea un cliente para interactuar con el servidor MLflow.

In [14]:
from mlflow.tracking import MlflowClient


client = MlflowClient("http://127.0.0.1:5000")

Listemos los modelos registrados actualmente (si ya corriste el notebook antes, puede que veas múltiples versiones):

In [15]:
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=mlflow.get_tracking_uri())
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1789327683506, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description='', last_updated_timestamp=1789327683524, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1789327683524, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1789327683524, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='4b1cd6e8e47143b8a6d01f8821026937', run_link='', source='models:/m-cae2614ce96847d4b2cbdef50f0cde31', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>], name='iris-classifier', tags={}, workspace='default'>]

## Model Registry: versiones y aliases (flujo recomendado)

En este escenario ya registramos el modelo usando `mlflow.sklearn.log_model(..., registered_model_name=...)`.

Ahora veremos cómo:
- listar versiones disponibles
- asignar un alias (por ejemplo `champion` / `candidate`)

> Nota: el uso de *stages* existe, pero MLflow recomienda cada vez más usar **aliases** para consumir modelos de forma estable.

In [16]:
model_name = "iris-classifier"

latest_versions = client.search_model_versions(f"name='{model_name}'")

[(mv.version, mv.current_stage, mv.run_id) for mv in latest_versions]

[('1', 'None', '4b1cd6e8e47143b8a6d01f8821026937')]

In [17]:
# Elegimos la versión más reciente (por creation timestamp)
latest_mv = max(latest_versions, key=lambda mv: int(mv.creation_timestamp))
latest_version = latest_mv.version

# Asignamos aliases típicos
client.set_registered_model_alias(model_name, "candidate", latest_version)
client.set_registered_model_alias(model_name, "champion", latest_version)

client.get_registered_model(model_name).aliases

{'candidate': '1', 'champion': '1'}

In [18]:
# (Opcional) Stage: requiere que el servidor tenga backend store (SQLite/Postgres/etc.)
# y el model registry habilitado. En local suele funcionar.

client.transition_model_version_stage(
    name=model_name,
    version=latest_version,
    stage="Staging",
    archive_existing_versions=False,
)

client.get_model_version(name=model_name, version=latest_version).current_stage

/var/folders/dp/2nhmtbqj49v3st8xl2stwglw0000gn/T/ipykernel_8127/3819451268.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


'Staging'

## **Model Registry vs Tracking: Diferencias Clave**

### **MLflow Tracking**
- **Propósito**: Seguimiento de experimentos y ejecuciones individuales
- **Almacena**: Parámetros, métricas, artifacts y metadatos de cada run
- **Uso**: Durante el desarrollo y experimentación
- **Organización**: Por experimentos y ejecuciones
- **Acceso**: A través de la API de MLflow o interfaz web

### **Model Registry**
- **Propósito**: Gestión del ciclo de vida completo de modelos en producción
- **Almacena**: Versiones de modelos, stages (Staging, Production), transiciones
- **Uso**: Para despliegue, versionado y gestión operacional
- **Organización**: Por nombre de modelo y versiones
- **Acceso**: A través del Model Registry API


## **Ventaja Principal del Model Registry**

**El Model Registry va más allá del tracking al proporcionar:**

1. **Versionado de Modelos**: Mantiene un historial completo de todas las versiones
2. **Stages de Despliegue**: Controla qué versión está en Staging vs Production
3. **Aprobaciones**: Permite flujos de trabajo para promocionar modelos
4. **Trazabilidad**: Conecta cada versión con su experimento original
5. **Operaciones**: Facilita rollbacks, comparaciones y auditorías

**En resumen**: Tracking es para experimentar, Registry es para producir y mantener modelos en el mundo real.

---


## **Diferencias Clave con el Escenario 1**

### **Arquitectura**
- **Escenario 1**: MLflow local con archivos
- **Escenario 2**: Servidor MLflow con base de datos SQLite

### **Colaboración**
- **Escenario 1**: Uso individual
- **Escenario 2**: Múltiples usuarios pueden acceder al mismo servidor

### **Persistencia**
- **Escenario 1**: Datos almacenados en archivos locales
- **Escenario 2**: Metadatos en base de datos SQLite, artifacts en sistema de archivos

### **Model Registry**
- **Escenario 1**: No disponible
- **Escenario 2**: Completamente funcional para gestión de modelos

---

## **Ventajas del Escenario 2**

1. **Colaboración**: Múltiples científicos de datos pueden trabajar en el mismo servidor
2. **Persistencia**: Los metadatos se mantienen en una base de datos estructurada
3. **Escalabilidad**: Fácil de migrar a bases de datos más robustas (PostgreSQL, MySQL)
4. **Model Registry**: Gestión completa del ciclo de vida de los modelos
5. **Acceso Web**: Interfaz web disponible en http://127.0.0.1:5000

---

## **Casos de Uso Apropiados**

- **Equipos pequeños**: 2-10 científicos de datos
- **Desarrollo local**: Entorno de desarrollo y testing
- **Prototipado**: Validación de conceptos antes de producción
- **Educación**: Cursos y talleres de MLOps
- **Investigación**: Experimentos colaborativos en academia

## Ejercicio 4 (equipos - sala de Zoom)

**Modalidad:** equipos de 2-3 personas, sala de breakout en Zoom.
**Tiempo sugerido:** 15 minutos.

Ya corrieron el Escenario 1 (sin servidor) y el Escenario 2 (con servidor local +
SQLite). En equipo:

1. Repitan el entrenamiento del Escenario 1 pero con un `random_state` distinto al que
   usaron sus companeros, y comparen visualmente los resultados en la MLflow UI de
   ambos escenarios (dos pestanas del navegador).
2. Si su universidad les diera un servidor MLflow compartido para todo el curso
   (equivalente al Escenario 2 pero para 30 estudiantes a la vez), ?que problema nuevo
   aparecería que no tenian en el Escenario 1? (piensen en nombres de experimentos,
   permisos, quien puede borrar un modelo registrado, etc.)
3. Propongan, en una frase, una convencion de nombres de experimentos para que el
   servidor compartido no se vuelva un caos (por ejemplo incluir el nombre del equipo).

**Entregable para la plenaria:** la convencion de nombres que propusieron en el punto 3.
